In [ ]:
from agents import Agent, Runner, SQLiteSession
from agents.mcp import MCPServerStdio
from dotenv import load_dotenv
import os
from openai import AsyncOpenAI
from IPython.display import display, Markdown
from pathlib import Path
from qdrant_client import QdrantClient
from agents.extensions.models.litellm_model import LitellmModel
import gradio as gr
load_dotenv(override=True)

In [ ]:
cerebras_api_key = os.getenv("CEREBRAS_API_KEY")
model = LitellmModel(model="cerebras/gpt-oss-120b", api_key=cerebras_api_key)

In [ ]:
agent = Agent("Tester", model=model)
response = await Runner.run(agent, "what is 2+2?")
print(response.final_output)

In [ ]:

urls = ["https://edwarddonner.com", "https://edwarddonner.com/curriculum", "https://edwarddonner.com/about"]

knowledge_dir = Path.cwd() / "knowledge"
knowledge_dir.mkdir(exist_ok=True)
vectordb_path = knowledge_dir / "vectordb"

fetch_params = {
    "command": "uvx",
    "args": ["mcp-server-fetch"],
}

vectorstore_params = {
    "command": "uvx",
    "args": ["mcp-server-qdrant"],
    "env": {
        "QDRANT_LOCAL_PATH": str(vectordb_path),
        "COLLECTION_NAME": "knowledge",
    },
}

In [ ]:
CONTEXT = """
You are an Agent with expert knowledge about Ed Donner with particular focus on his online AI courses.
"""

INSTRUCTIONS = CONTEXT + """
You are populating your memories with information retrieved from a given website.
Use your MCP tools to retrieve the website. Extract key knowledge. Check what's already in your memories to avoid duplicates.
After you are done, reply with a brief status update and the number of memories you added.
Aim to add at least 10 unique memories, unless your existing memories are already comprehensive.
"""

In [ ]:
async with MCPServerStdio(params=fetch_params, client_session_timeout_seconds=120) as fetch_mcp:
    tools = await fetch_mcp.list_tools()
tools

In [ ]:
async with MCPServerStdio(params=vectorstore_params, client_session_timeout_seconds=120) as vectorstore_mcp:
    tools = await vectorstore_mcp.list_tools()
tools

In [ ]:
for url in (urls * 3)[:1]:
    async with MCPServerStdio(params=fetch_params, client_session_timeout_seconds=120) as fetch_mcp:
        async with MCPServerStdio(params=vectorstore_params, client_session_timeout_seconds=120) as vectorstore_mcp:
            agent = Agent(name="Ingester", model=model, instructions=INSTRUCTIONS, mcp_servers=[fetch_mcp, vectorstore_mcp])
            task = f"Add unique memories with information from this website: {url} and reply with a one sentence status update including how many memories were added."
            response = await Runner.run(agent, task, max_turns=50)
            display(Markdown(response.final_output))

In [ ]:
client = QdrantClient(path=str(vectordb_path))
collection_name = "knowledge"

info = client.get_collection(collection_name)
print(f"Memories in '{collection_name}': {info.points_count}\n")

points, _ = client.scroll(collection_name=collection_name, limit=200, with_payload=True, with_vectors=False)
for i, p in enumerate(points, 1):
    doc = (p.payload or {}).get("document", "")
    preview = doc.replace("\n", " ")[:160]
    print(f"{i:>3}. {preview}{'...' if len(doc) > 160 else ''}")

client.close()

In [ ]:
EXPERT_INSTRUCTIONS = """
You are an expert about Ed Donner and his online courses. You are answering questions about him and his courses to visitors on his website.
Use your memories to help answer the question. If you don't know the answer, say you don't know.
"""

In [ ]:
convo = SQLiteSession("test_conversation")

In [ ]:
async def chat(message, history):
    async with MCPServerStdio(params=vectorstore_params, client_session_timeout_seconds=120) as vectorstore_mcp:
        agent = Agent(name="Expert", model=model, instructions=EXPERT_INSTRUCTIONS, mcp_servers=[vectorstore_mcp])
        response = await Runner.run(agent, message, session=convo)
        return response.final_output


In [ ]:
gr.ChatInterface(chat).launch()